# 07 · Random Forest

**What this notebook covers**

- Randomised search for both feature sets
- Top candidates and overfitting check
- Feature importance: impurity vs permutation

## 1. Environment setup
### 1.1 Imports
Standard model-notebook imports plus the estimator-specific ones.

In [ ]:
import hashlib
import json
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score, brier_score_loss, f1_score, make_scorer,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

### 1.2 Configuration
Every tunable knob lives here. Lower the search budget for a quick dry run, but use
the defaults for the reported results.

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_PREFIX = "07"
NOTEBOOK_NAME = "07_random_forest.ipynb"
CV_FOLDS = 5

VARIANTS = ["day_zero", "full_lifecycle"]
N_ITER = 25                  # random-search candidates per variant
RF_SPACE = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [None, 12, 20, 30],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.3, 0.5],
    "model__class_weight": ["balanced", "balanced_subsample"],
}
PERMUTATION_SAMPLE = 5000    # Dev rows used for permutation importance (speed)
PERMUTATION_REPEATS = 5

### 1.3 Project paths, shared constants and helpers

In [ ]:
# Resolve the repository root regardless of where the kernel was started
# (VS Code starts kernels in notebooks/, Jupyter may start them at the repo root).
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "DECISIONS.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (no DECISIONS.md in any parent directory).")


PROJECT_ROOT = find_project_root()

PATHS = {
    "raw": PROJECT_ROOT / "data" / "hotel_bookings.csv",
    "clean": PROJECT_ROOT / "data" / "interim" / "hotel_bookings_clean.csv",
    "splits": PROJECT_ROOT / "data" / "processed" / "splits",
    "features": PROJECT_ROOT / "data" / "processed" / "features",
    "artifacts": PROJECT_ROOT / "artifacts",
    "preprocessors": PROJECT_ROOT / "artifacts" / "preprocessors",
    "model_outputs": PROJECT_ROOT / "artifacts" / "model_outputs",
    "models": PROJECT_ROOT / "models",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "tables": PROJECT_ROOT / "reports" / "tables",
    "notebooks": PROJECT_ROOT / "notebooks",
}
for key in ("artifacts", "preprocessors", "model_outputs", "models", "figures", "tables"):
    PATHS[key].mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Columns that must always be treated as categorical strings when re-reading CSVs.
STRING_COLUMNS = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "agent", "company",
    "customer_type", "arrival_season",
]
TARGET = "is_canceled"
ID_COLUMN = "booking_id"
LEAKAGE_COLUMNS = ["reservation_status", "reservation_status_date"]

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Streaming SHA-256 of a file (used to fingerprint every artifact)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path.relative_to(PROJECT_ROOT)}")
    return json.loads(path.read_text())


def read_frame(path: Path) -> pd.DataFrame:
    """Read a project CSV with categorical columns forced to plain Python strings.

    `keep_default_na=False` + an explicit NA list stops pandas from turning legitimate
    category labels into NaN; numeric columns are parsed normally.
    """
    if not path.exists():
        raise FileNotFoundError(f"Required data file missing: {path.relative_to(PROJECT_ROOT)}")
    frame = pd.read_csv(
        path,
        dtype={col: object for col in STRING_COLUMNS},
        keep_default_na=False,
        na_values=["", "NaN", "nan"],
    )
    present = [c for c in STRING_COLUMNS if c in frame.columns]
    frame[present] = frame[present].astype(object)
    return frame

In [ ]:
# One visual language for every figure in the project:
#  - blue = not cancelled, orange = cancelled (validated colour-blind-safe pair)
#  - single-series charts use blue; recessive grey grid; no top/right spines; no dual axes.
COLORS = {
    "not_canceled": "#2a78d6",
    "canceled": "#eb6834",
    "neutral": "#8a8985",
    "ink": "#0b0b0b",
    "ink_muted": "#52514e",
}
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQUENTIAL_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_diverging", ["#2a78d6", "#e8e7e3", "#eb6834"]
)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "axes.prop_cycle": mpl.cycler(color=SERIES_COLORS),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.labelcolor": COLORS["ink_muted"],
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.titlecolor": COLORS["ink"],
    "axes.grid": True,
    "grid.color": "#e8e7e3",
    "grid.linewidth": 0.8,
    "xtick.color": COLORS["ink_muted"],
    "ytick.color": COLORS["ink_muted"],
    "lines.linewidth": 2,
    "legend.frameon": False,
})


def save_figure(fig, name: str) -> Path:
    """Save a figure under reports/figures with this notebook's numeric prefix."""
    path = PATHS["figures"] / f"{NOTEBOOK_PREFIX}_{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved figure -> {path.relative_to(PROJECT_ROOT)}")
    return path

In [ ]:
import importlib.metadata as importlib_metadata

def library_versions(packages=("numpy", "pandas", "scikit-learn", "xgboost", "matplotlib", "seaborn", "scipy", "shap", "joblib")) -> dict:
    versions = {"python": sys.version.split()[0], "platform": platform.platform()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not installed"
    return versions

LIBRARY_VERSIONS = library_versions()
pd.Series(LIBRARY_VERSIONS, name="version").to_frame()

## 2. Load Train and Dev features (contract-verified)
Inputs are hash-checked against `feature_spec.json`. The held-out evaluation split is
deliberately not loaded here.

In [ ]:
# Load the shared feature contract and ONLY the Train and Dev feature files.
# The held-out split is intentionally never referenced in model notebooks
# (the validation gate in notebook 10 scans this notebook's source to prove it).
FEATURE_SPEC = load_json(PATHS["artifacts"] / "feature_spec.json")

TRAIN_PATH = PATHS["features"] / "train.csv"
DEV_PATH = PATHS["features"] / "dev.csv"
DATA_FILES_READ = [str(TRAIN_PATH.relative_to(PROJECT_ROOT)), str(DEV_PATH.relative_to(PROJECT_ROOT))]

INPUT_HASHES = {"train": sha256_file(TRAIN_PATH), "dev": sha256_file(DEV_PATH)}
for split_name, digest in INPUT_HASHES.items():
    expected = FEATURE_SPEC["file_hashes"][split_name]
    assert digest == expected, (
        f"{split_name} feature file hash mismatch - re-run notebook 04 before training models."
    )

train_df = read_frame(TRAIN_PATH)
dev_df = read_frame(DEV_PATH)

X_train = train_df.drop(columns=[TARGET, ID_COLUMN])
y_train = train_df[TARGET].astype(int)
X_dev = dev_df.drop(columns=[TARGET, ID_COLUMN])
y_dev = dev_df[TARGET].astype(int)
dev_ids = dev_df[ID_COLUMN]

assert not set(LEAKAGE_COLUMNS) & set(X_train.columns), "Leakage columns reached the model inputs."
print(f"Train: {X_train.shape}, positive rate {y_train.mean():.4f}")
print(f"Dev:   {X_dev.shape}, positive rate {y_dev.mean():.4f}")

### 2.1 Shared model-notebook helpers
These do the same job in notebooks 05–09: the CV splitter, multi-metric scoring
(refit on PR-AUC), fold-score extraction and `persist_model_outputs`, which writes the
model card contract consumed by the validation gate and the evaluation notebooks.

In [ ]:
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
SCALE_POS_WEIGHT = float((y_train == 0).sum() / (y_train == 1).sum())


def build_scoring(include_probability_metrics: bool = True) -> dict:
    """Multi-metric scoring used by every model notebook (refit metric = PR-AUC)."""
    scoring = {
        "pr_auc": "average_precision",
        "roc_auc": "roc_auc",
        "f1": make_scorer(f1_score, zero_division=0),
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": make_scorer(recall_score, zero_division=0),
        "balanced_accuracy": "balanced_accuracy",
        "accuracy": "accuracy",
    }
    if include_probability_metrics:
        scoring["neg_brier"] = "neg_brier_score"
    return scoring


def load_preprocessor(family: str, variant: str):
    """Load an UNFITTED preprocessing template produced by notebook 04."""
    path = PATHS["preprocessors"] / f"preprocessor__{family}__{variant}.joblib"
    return clone(joblib.load(path))


def make_pipeline(estimator, family: str, variant: str) -> Pipeline:
    return Pipeline([("preprocess", load_preprocessor(family, variant)), ("model", estimator)])


def fold_scores_from_search(search, n_splits: int) -> pd.DataFrame:
    """Per-fold CV scores of the best candidate, read from cv_results_ (no refit needed)."""
    results, best = search.cv_results_, search.best_index_
    metrics = sorted({key.split("_test_", 1)[1] for key in results if key.startswith("split0_test_")})
    rows = []
    for fold in range(n_splits):
        row = {"fold": fold}
        for metric in metrics:
            row[metric] = results[f"split{fold}_test_{metric}"][best]
            train_key = f"split{fold}_train_{metric}"
            if train_key in results:
                row[f"train_{metric}"] = results[train_key][best]
        rows.append(row)
    return tidy_fold_scores(pd.DataFrame(rows))


def fold_scores_from_cross_validate(cv_output: dict) -> pd.DataFrame:
    frame = pd.DataFrame({
        key.replace("test_", "", 1) if key.startswith("test_") else key: values
        for key, values in cv_output.items() if key.startswith(("test_", "train_"))
    })
    frame.insert(0, "fold", range(len(frame)))
    return tidy_fold_scores(frame)


def tidy_fold_scores(frame: pd.DataFrame) -> pd.DataFrame:
    for prefix in ("", "train_"):
        if f"{prefix}neg_brier" in frame:
            frame[f"{prefix}brier"] = -frame.pop(f"{prefix}neg_brier")
    return frame


def summarise_folds(fold_scores: pd.DataFrame) -> dict:
    metric_cols = [c for c in fold_scores.columns if c != "fold" and not c.startswith("train_")]
    return {m: {"mean": float(fold_scores[m].mean()), "std": float(fold_scores[m].std(ddof=1))} for m in metric_cols}


def quick_dev_metrics(y_true, y_proba) -> dict:
    """Lightweight sanity metrics only - the official evaluation happens in notebook 11."""
    return {
        "pr_auc": float(average_precision_score(y_true, y_proba)),
        "roc_auc": float(roc_auc_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


def persist_model_outputs(*, model_key, variant, family, pipeline, fold_scores, dev_proba,
                          best_params, imbalance_strategy, fit_seconds, search=None,
                          cv_rows=None, extra=None) -> dict:
    """Write the model + the standard model-card contract consumed by notebooks 10-14."""
    model_id = f"{model_key}__{variant}"
    out_dir = PATHS["model_outputs"] / model_id
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = PATHS["models"] / f"{model_id}.joblib"
    joblib.dump(pipeline, model_path)

    fold_scores.to_csv(out_dir / "cv_fold_scores.csv", index=False)
    pd.DataFrame({ID_COLUMN: dev_ids.values, "y_true": y_dev.values, "y_proba": dev_proba}).to_csv(
        out_dir / "dev_predictions.csv", index=False
    )
    if search is not None:
        pd.DataFrame(search.cv_results_).drop(columns=["params"], errors="ignore").to_csv(
            out_dir / "search_results.csv", index=False
        )

    card = {
        "model_id": model_id,
        "model_key": model_key,
        "variant": variant,
        "notebook": NOTEBOOK_NAME,
        "preprocessor_family": family,
        "imbalance_strategy": imbalance_strategy,
        "cv": {
            "type": type(CV).__name__,
            "n_splits": CV.get_n_splits(),
            "shuffle": bool(CV.shuffle),
            "random_state": CV.random_state,
            "n_rows": int(cv_rows if cv_rows is not None else len(X_train)),
        },
        "n_fit_rows": int(len(X_train)),
        "n_dev_rows": int(len(X_dev)),
        "data_files_read": DATA_FILES_READ,
        "input_hashes": INPUT_HASHES,
        "best_params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) for k, v in best_params.items()},
        "cv_summary": summarise_folds(fold_scores),
        "dev_quick": quick_dev_metrics(y_dev, dev_proba),
        "fit_seconds": round(float(fit_seconds), 2),
        "model_path": str(model_path.relative_to(PROJECT_ROOT)),
        "model_sha256": sha256_file(model_path),
        "random_state": RANDOM_STATE,
        "library_versions": LIBRARY_VERSIONS,
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **(extra or {}),
    }
    save_json(card, out_dir / "model_card.json")
    print(f"[{model_id}] CV PR-AUC {card['cv_summary']['pr_auc']['mean']:.4f} "
          f"± {card['cv_summary']['pr_auc']['std']:.4f} | Dev PR-AUC {card['dev_quick']['pr_auc']:.4f}")
    return card

## 3. Randomised search per variant
The space has 288 combinations; 25 random candidates × 5 folds cover it well (random
search finds near-optimal settings in far fewer fits than a grid, per Bergstra &
Bengio 2012). Parallelism sits inside the forest (`n_jobs=-1`) rather than across
candidates, to avoid CPU over-subscription.

In [ ]:
rf_searches, rf_cards = {}, {}
for variant in VARIANTS:
    pipeline = make_pipeline(RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE), "tree", variant)
    search = RandomizedSearchCV(pipeline, RF_SPACE, n_iter=N_ITER, scoring=build_scoring(), refit="pr_auc", cv=CV,
                                n_jobs=1, random_state=RANDOM_STATE, return_train_score=True)
    start = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    print(f"[{variant}] best params {search.best_params_} ({elapsed / 60:.1f} min)")
    rf_searches[variant] = search
    rf_cards[variant] = persist_model_outputs(
        model_key="random_forest", variant=variant, family="tree", pipeline=search.best_estimator_,
        fold_scores=fold_scores_from_search(search, CV_FOLDS),
        dev_proba=search.best_estimator_.predict_proba(X_dev)[:, 1],
        best_params=search.best_params_, imbalance_strategy="class_weight", fit_seconds=elapsed, search=search,
    )

## 4. Search diagnostics
### 4.1 Top candidates
If the top candidates sit within one standard deviation of each other, the choice
between them is noise, so prefer the cheaper or shallower one.

In [ ]:
columns = ["rank_test_pr_auc", "mean_test_pr_auc", "std_test_pr_auc", "mean_train_pr_auc", "mean_fit_time",
           "param_model__n_estimators", "param_model__max_depth", "param_model__min_samples_leaf",
           "param_model__max_features", "param_model__class_weight"]
pd.DataFrame(rf_searches["day_zero"].cv_results_)[columns].sort_values("rank_test_pr_auc").head(10).round(4)

### 4.2 Over-fitting check
Random forests reach near-perfect Train scores by design. What matters is that the
validation score is stable across folds (low SD).

In [ ]:
pd.DataFrame({
    variant: {"cv_pr_auc_mean": card["cv_summary"]["pr_auc"]["mean"], "cv_pr_auc_std": card["cv_summary"]["pr_auc"]["std"],
              "dev_pr_auc": card["dev_quick"]["pr_auc"]}
    for variant, card in rf_cards.items()
}).T.round(4)

## 5. Feature importance: impurity vs permutation (day-zero)
### 5.1 Impurity-based importance (on encoded features)

In [ ]:
best_rf = rf_searches["day_zero"].best_estimator_
impurity = pd.Series(best_rf.named_steps["model"].feature_importances_,
                     index=best_rf.named_steps["preprocess"].get_feature_names_out(), name="impurity_importance")
impurity.sort_values(ascending=False).to_csv(PATHS["tables"] / "07_random_forest_day_zero_impurity_importance.csv")
impurity.sort_values(ascending=False).head(15).round(4).to_frame()

### 5.2 Permutation importance (on raw input columns, Dev subsample)
Permutation importance measures the drop in Dev PR-AUC when one *original* column is
shuffled. It runs through the whole pipeline, so a column's one-hot children are
permuted together. Unlike impurity importance, it is not biased toward
high-cardinality features.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_dev), size=min(PERMUTATION_SAMPLE, len(X_dev)), replace=False)
used_columns = [c for cols in FEATURE_SPEC["variants"]["day_zero"].values() for c in cols]
perm = permutation_importance(best_rf, X_dev.iloc[sample_idx], y_dev.iloc[sample_idx], scoring="average_precision",
                              n_repeats=PERMUTATION_REPEATS, random_state=RANDOM_STATE, n_jobs=-1)
permutation = pd.DataFrame({"feature": X_dev.columns, "mean_drop_pr_auc": perm.importances_mean,
                            "std": perm.importances_std})
permutation = permutation[permutation["feature"].isin(used_columns)].sort_values("mean_drop_pr_auc", ascending=False)
permutation.to_csv(PATHS["tables"] / "07_random_forest_day_zero_permutation_importance.csv", index=False)
permutation.head(15).round(4)

### 5.3 Permutation importance chart
Error bars show ±1 SD across the permutation repeats. Features whose bar crosses zero
carry no reliable signal.

In [ ]:
top = permutation.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["feature"], top["mean_drop_pr_auc"], xerr=top["std"], color=COLORS["not_canceled"], height=0.65,
        error_kw={"ecolor": COLORS["ink_muted"], "elinewidth": 1, "capsize": 2})
ax.axvline(0, color=COLORS["ink_muted"], linewidth=1)
ax.set_xlabel("Drop in Dev PR-AUC when the column is shuffled")
ax.set_title("Random Forest (day-zero): permutation importance", loc="left")
ax.grid(axis="y", visible=False)
save_figure(fig, "permutation_importance")
plt.show()

## 6. Hand-off summary
- `random_forest__day_zero` and `random_forest__full_lifecycle` are logged.
- Importance tables (`reports/tables/07_*importance*.csv`) are used in notebook 13.
- Confirmed for the validation gate: imbalance is handled by `class_weight` at fit time
  (no row duplication or resampling).